# 02 — Event deduplication and cutoff

**Objective.** Build deterministic company, funding-event, investor-edge, and acquisition tables under the frozen duplicate and administrative-cutoff rules.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

Determinism rule: no row-order-dependent `GroupBy.first()` operation is used.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("02", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
from cruxvc.events import build_event_tables
from cruxvc.io import read_json, write_json, write_table
from cruxvc.schema import read_source_csv

raw_dir = P.raw / "crunchbase_october_2013"
paths = {
    "companies": raw_dir / "crunchbase-companies.csv",
    "rounds": raw_dir / "crunchbase-rounds.csv",
    "investments": raw_dir / "crunchbase-investments.csv",
    "acquisitions": raw_dir / "crunchbase-acquisitions.csv",
}
CTX.recorder.inputs.extend(paths.values())
tables = {name: read_source_csv(path) for name, path in paths.items()}

In [ ]:
result = build_event_tables(
    tables["companies"], tables["rounds"], tables["investments"], tables["acquisitions"],
    administrative_cutoff=CFG["source"]["administrative_cutoff"],
)
companies_path = write_table(result.companies, P.interim / "companies_canonical.parquet")
funding_path = write_table(result.funding_events, P.interim / "funding_events.parquet")
investments_path = write_table(result.investment_edges, P.interim / "investment_edges.parquet")
acquisitions_path = write_table(result.acquisitions, P.interim / "acquisitions_clean.parquet")
audit_path = write_json(result.audit, P.audits / "02_event_cleaning_audit.json")

In [ ]:
source_manifest = read_json(P.protocol / "source_manifest.json")
expected = CFG["source"]["expected_clean_counts"]
if source_manifest["all_expected_hashes_match"] and CFG["execution"]["strict_expected_counts_when_hashes_match"]:
    observed_round_distinct = result.audit["raw_round_rows"] - result.audit["exact_round_duplicates"]
    if observed_round_distinct != int(expected["funding_rounds_distinct"]):
        raise RuntimeError(f"Exact round deduplication did not reproduce {expected['funding_rounds_distinct']}: observed {observed_round_distinct}")
    if result.audit["valid_unique_investment_edges"] != int(expected["investments_valid_unique"]):
        raise RuntimeError(f"Investment key deduplication did not reproduce {expected['investments_valid_unique']}: observed {result.audit['valid_unique_investment_edges']}")

In [ ]:
CTX.recorder.complete([companies_path, funding_path, investments_path, acquisitions_path, audit_path])
print(result.audit)